In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration    
CONFIG = Configuration()

In [ ]:
import numpy as np

x_train = np.load("../data/ViolaJones/x_train.npy")
x_test = np.load("../data/ViolaJones/x_test.npy")


x = np.concatenate((x_train, x_test), axis=0)

# np.save("../data/ViolaJones/vpc_faces.npy", x)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

no_faces_folder = "../data/ViolaJones/no_faces/vpc"
output_dir = Path(no_faces_folder)
output_dir.mkdir(parents=True, exist_ok=True)

for i, img in enumerate(x):
    arr = np.asarray(img)

    # Handle channel-first arrays: (C, H, W) -> (H, W, C)
    if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
        arr = np.moveaxis(arr, 0, -1)

    # Convert to uint8 image range [0, 255]
    if arr.dtype != np.uint8:
        if np.issubdtype(arr.dtype, np.floating):
            if arr.size > 0 and arr.max() <= 1.0 and arr.min() >= 0.0:
                arr = (arr * 255.0).clip(0, 255).astype(np.uint8)
            else:
                arr = arr.clip(0, 255).astype(np.uint8)
        else:
            arr = arr.clip(0, 255).astype(np.uint8)

    # If image has shape (H, W, 1), save as grayscale 2D
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr[:, :, 0]

    file_path = output_dir / f"no_face_{i:06d}.png"
    plt.imsave(file_path, arr, cmap="gray" if arr.ndim == 2 else None)

print(f"Saved {len(x)} images to {output_dir.resolve()}")

# Original dataset passing Open CV model

In [ ]:

import os

import shutil
import cv2
from maikol_utils.file_utils import list_dir_files, make_dirs

cascade_path = os.path.join(CONFIG.cv_haar_cascades, "haarcascade_frontalface_default.xml")
face_cascade = cv2.CascadeClassifier(str(cascade_path))
if face_cascade.empty():
    raise FileNotFoundError(f"Could not load cascade XML: {cascade_path}")

# Output folders: images that pass vs fail the detector
passed_dir = os.path.join(CONFIG.faces_path, "cv_passed")
failed_dir = os.path.join(CONFIG.faces_path, "cv_failed")
make_dirs([passed_dir, failed_dir])

image_paths, _ = list_dir_files(CONFIG.faces_path)

passed = []
failed = []
unreadable = []

for img_path in image_paths:
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if img is None:
        unreadable.append(img_path)
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(24, 24)
    )

    target_dir = passed_dir if len(faces) > 0 else failed_dir
    shutil.copy2(img_path, os.path.join(target_dir, os.path.basename(img_path)))

    if len(faces) > 0:
        passed.append(img_path)
    else:
        failed.append(img_path)

print(f"Source dir: {CONFIG.faces_path}")
print(f"Total candidate images: {len(image_paths)}")
print(f"Passed (>=1 face): {len(passed)}")
print(f"Failed (0 faces): {len(failed)}")
print(f"Unreadable: {len(unreadable)}")
print(f"Passed output: {passed_dir}")
print(f"Failed output: {failed_dir}")

# Keep lists available for downstream analysis in the notebook
passed_paths = [str(p) for p in passed]
failed_paths = [str(p) for p in failed]
unreadable_paths = [str(p) for p in unreadable]

# Create partition

In [ ]:
import numpy as np
CONFIG = Configuration(
    max_bg_samples=500,
    stride=8,
    
)
rng = np.random.default_rng(seed=42)


In [ ]:
from maikol_utils.file_utils import list_dir_files


all_faces, n = list_dir_files(CONFIG.faces_all_path)
print(f"Total images in faces directory: {n}")

all_faces = rng.permutation(all_faces)

test_faces = all_faces[:int(CONFIG.test_size * len(all_faces))]
train_faces = all_faces[int(CONFIG.test_size * len(all_faces)):]

print(f"Train faces: {len(train_faces)}")
print(f"Test faces: {len(test_faces)}")

In [ ]:
from tqdm import tqdm 
import shutil
import os

for split, paths in [("train", train_faces), ("test", test_faces)]:
    output_dir = CONFIG.faces_train_path if split == "train" else CONFIG.faces_test_path
    for img_path in tqdm(paths, desc=f"Copying {split} images"):
        shutil.copy2(img_path, os.path.join(output_dir, os.path.basename(img_path)))
    print(f"Copied {len(paths)} images to {output_dir}")

# No faces dataset

In [ ]:
from src.data import load_gb_images
from maikol_utils.file_utils import list_dir_files

# test_bg = load_gb_images(CONFIG, partition="test")
test_bg = list_dir_files('../data/ViolaJones/no_faces/open-images-v7/test/data')[0]

In [ ]:
from tqdm import tqdm 
from src.data import get_all_image_crops

all_crops = []

for img_path in tqdm(test_bg, desc="Extracting crops from background images"):
    crops = get_all_image_crops(CONFIG, img_path)
    all_crops.extend(crops)

len(all_crops)

In [ ]:
all_crops = rng.choice(all_crops, 100_000, replace=False)
all_crops = [crop['img'] for crop in all_crops]

In [ ]:
all_crops[5].shape

In [ ]:
import os
import matplotlib.pyplot as plt

for idx, crop in tqdm(enumerate(all_crops), total=len(all_crops), desc="Saving crops to disk"):
    # safe on disk
    path = os.path.join(CONFIG.no_facescrops_path, f"crop_{idx:06d}.png")
    plt.imsave(path, crop)